# 04a v12 — ECG Forecasting: Encoder-Decoder CNN + Composite Loss

## Why v11 produced PRD ≈ 100% (total failure)

Three compounding problems were diagnosed from v11's outputs:

| Problem | Evidence | Fix in v12 |
|---|---|---|
| **Wrong architecture for forecasting** | Output position `t` could only see input `0..t` (causal). For a future-horizon task, output[0] must see ALL 490 input steps. The model learned to reconstruct input, not forecast. | Encoder-Decoder: CNN encoder compresses 490→latent, decoder generates 490 future steps from global context |
| **Data was pre-normalized** | IQR≈1.0 on all leads means data was already in [-3,3] range. Per-lead IQR scaling did nothing (divided by ≈1.0), just added dead code. | Removed redundant per-lead normalization; kept z-score on train stats |
| **GradientMAELoss grad_weight too low** | Notebook itself warned `raise GRAD_WEIGHT to 3.0-4.0` — v11 shipped with 2.0 anyway. With std≈1.16 on normalized data, the gradient term was underpowered. | GRAD_WEIGHT=4.0; added frequency-domain loss term (FFT magnitude) to force QRS morphology at the right temporal scale |

### Architecture change: causal dilated CNN → Encoder-Decoder
- **Encoder**: 5 dilated-CNN blocks compress `(B,12,490)` → `(B,C,1)` via adaptive pooling → global context
- **Decoder**: learned positional query + cross-attention-style MLP, then 5 upsampling CNN blocks to produce `(B,490,12)`
- Receptive field covers the full input window, and every output step conditions on the ENTIRE input — correct for forecasting


In [1]:
# CELL 1 - IMPORTS
import os, pickle, warnings
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.signal import find_peaks
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("OK Imports ready")


PyTorch : 2.11.0
Device  : cpu
OK Imports ready


In [2]:
# CELL 2 - LOAD DATA + NORMALISATION
#
# v11 BUG: Per-lead IQR normalisation was a no-op because the PTB-XL
# pipeline already clips/scales to [-3, 3] mV (std\u22481.17, IQR\u22481.0 on all
# leads). Dividing by \u22481.0 changed nothing useful; the dead code just
# obscured the real problem.
#
# v12 FIX: Simple global z-score (mean/std computed on train concat of X+y).
# This keeps scale consistent without the IQR illusion. If your data is
# already clipped to [-3,3] you'll get mean\u22480.08, std\u22481.17 -> normalised
# to near unit Gaussian. Metrics are de-normalised back to mV before reporting.

SAVE_DIR = os.path.join('..', 'data', 'processed')
FIG_DIR  = os.path.join('..', 'reports', 'figures', 'cnn_v12')
CKPT_DIR = os.path.join('..', 'reports', 'checkpoints')

os.makedirs(FIG_DIR,  exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"Working dir : {os.getcwd()}")
print(f"Data dir    : {SAVE_DIR}")

X_train = np.load(os.path.join(SAVE_DIR, 'X_train.npy'))
y_train = np.load(os.path.join(SAVE_DIR, 'y_train.npy'))
X_val   = np.load(os.path.join(SAVE_DIR, 'X_val.npy'))
y_val   = np.load(os.path.join(SAVE_DIR, 'y_val.npy'))
X_test  = np.load(os.path.join(SAVE_DIR, 'X_test.npy'))
y_test  = np.load(os.path.join(SAVE_DIR, 'y_test.npy'))

with open(os.path.join(SAVE_DIR, 'config.pkl'), 'rb') as f:
    cfg = pickle.load(f)

LEAD_NAMES = cfg['lead_names']
FS         = cfg['sampling_rate']
INPUT_LEN  = cfg['input_len']
HORIZON    = cfg['horizon']
N_LEADS    = cfg['n_leads']

print(f"X_train : {X_train.shape}   y_train : {y_train.shape}")
print(f"y_train  mean={y_train.mean():.4f}  std={y_train.std():.4f}  "
      f"min={y_train.min():.3f}  max={y_train.max():.3f}")

# ------------------------------------------------------------------
# Normalisation: fit mean/std on train only, apply everywhere.
# If your preprocessing already normalised the signal the std will be
# close to 1 and this is a near-no-op — that is fine, it is safe.
# ------------------------------------------------------------------
def fit_norm(X, y):
    """Compute global mean/std from flattened train X and y."""
    vals = np.concatenate([X.ravel(), y.ravel()])
    mu, sigma = vals.mean(), vals.std()
    sigma = max(sigma, 1e-6)
    return float(mu), float(sigma)

NORM_MU, NORM_SIGMA = fit_norm(X_train, y_train)

def normalize(arr):
    return (arr - NORM_MU) / NORM_SIGMA

def denormalize(arr):
    return arr * NORM_SIGMA + NORM_MU

X_train = normalize(X_train)
y_train = normalize(y_train)
X_val   = normalize(X_val)
y_val   = normalize(y_val)
X_test  = normalize(X_test)
y_test  = normalize(y_test)

print(f"\nNorm stats (train): mu={NORM_MU:.5f}  sigma={NORM_SIGMA:.5f}")
print(f"Normalised y_train: mean={y_train.mean():.4f} std={y_train.std():.4f}")
print("OK Data loaded and normalised")


Working dir : /Users/muhammadfahadsiddiqui/HIS Project/notebooks
Data dir    : ../data/processed
X_train : (31362, 490, 12)   y_train : (31362, 490, 12)
y_train  mean=0.0834  std=1.1706  min=-3.000  max=3.000

Norm stats (train): mu=0.08268  sigma=1.16667
Normalised y_train: mean=0.0006 std=1.0034
OK Data loaded and normalised


In [3]:
# CELL 3 - DATASET + AUGMENTATION
#
# Input  X: (N, T_in, leads) -> permuted to (leads, T_in) for the encoder
# Target y: (N, T_out, leads) -> kept as (T_out, leads)
#
# Augmentation: mild amplitude jitter only (±5%). No additive noise
# (destroys QRS morphology). No time-warping (changes RR intervals,
# confuses the forecaster about where the next beat lands).

class ECGForecastDataset(Dataset):
    def __init__(self, X, y, augment=False):
        # X: (N, T_in, leads)  ->  store as-is, permute in __getitem__
        # y: (N, T_out, leads)
        self.X       = torch.from_numpy(np.asarray(X, dtype=np.float32))
        self.y       = torch.from_numpy(np.asarray(y, dtype=np.float32))
        self.augment = augment

    def __len__(self): return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx].clone()   # (T_in, leads)
        y = self.y[idx].clone()   # (T_out, leads)
        if self.augment:
            scale = torch.empty(1).uniform_(0.95, 1.05)   # ±5% amplitude jitter
            x, y  = x * scale, y * scale
        # Encoder expects (leads, T_in)
        x = x.permute(1, 0)       # (leads, T_in)
        return x, y                # y stays (T_out, leads)


def make_loaders(X_tr, y_tr, X_v, y_v, X_te, y_te,
                 batch_train=64, batch_eval=128):
    kw = dict(num_workers=0, pin_memory=(DEVICE.type == 'cuda'))
    tr = DataLoader(ECGForecastDataset(X_tr, y_tr, augment=True),
                    batch_size=batch_train, shuffle=True, drop_last=True, **kw)
    vl = DataLoader(ECGForecastDataset(X_v,  y_v,  augment=False),
                    batch_size=batch_eval,  shuffle=False, **kw)
    te = DataLoader(ECGForecastDataset(X_te, y_te, augment=False),
                    batch_size=batch_eval,  shuffle=False, **kw)
    return tr, vl, te


cnn_tr, cnn_vl, cnn_te = make_loaders(
    X_train, y_train, X_val, y_val, X_test, y_test,
    batch_train=64, batch_eval=128)

xb, yb = next(iter(cnn_tr))
print(f"Train batch : x={xb.shape}  y={yb.shape}")
print(f"Batches     : train={len(cnn_tr)} | val={len(cnn_vl)} | test={len(cnn_te)}")
print("OK DataLoaders ready  (batch=64, ±5% jitter only, no noise injection)")


Train batch : x=torch.Size([64, 12, 490])  y=torch.Size([64, 490, 12])
Batches     : train=490 | val=31 | test=18
OK DataLoaders ready  (batch=64, ±5% jitter only, no noise injection)


In [4]:
# CELL 4 - MODEL ARCHITECTURE: CNN Encoder-Decoder
#
# WHY THE OLD CAUSAL CNN FAILED FOR FORECASTING
# -----------------------------------------------
# The v11 architecture used a causal dilated CNN where output position t
# could only attend to input positions 0..t. This is correct for autoregressive
# sequence modelling (language, speech). It is WRONG for ECG forecasting where:
#   - Output[0]  = first sample of the FUTURE window
#   - It must be conditioned on ALL 490 input samples (the past window)
#   - Causality in the decoder direction is the wrong inductive bias
#
# The result: the model learned a causal reconstruction of its own input,
# giving PRD≈100% (orthogonal to the future target).
#
# v12 ENCODER-DECODER
# --------------------
# Encoder: dilated CNN on past window -> global context vector
#   (B, leads, T_in) -> (B, C, T_in) -> AdaptiveAvgPool -> (B, C)
# Decoder: MLP projects context to (B, C, T_out) -> upsample CNN -> (B, T_out, leads)
#   Every output step sees the full encoded past. No causal masking in decoder.
#
# This is a seq2seq design: compress past into a summary, generate future from that summary.

class ResBlock1D(nn.Module):
    """Standard dilated residual block — no causal masking."""
    def __init__(self, channels, dilation=1, kernel_size=5, dropout=0.1):
        super().__init__()
        pad = (kernel_size - 1) * dilation // 2   # same-padding (non-causal)
        self.conv1 = nn.Conv1d(channels, channels, kernel_size,
                               dilation=dilation, padding=pad, bias=False)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size,
                               dilation=dilation, padding=pad, bias=False)
        self.gn1   = nn.GroupNorm(8, channels)
        self.gn2   = nn.GroupNorm(8, channels)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        h = self.drop(F.gelu(self.gn1(self.conv1(x))))
        h = self.gn2(self.conv2(h))
        return F.gelu(x + h)


class ECGEncoderDecoder(nn.Module):
    """
    Encoder-Decoder for ECG forecasting.
    Input  : (B, leads, T_in)     e.g. (B, 12, 490)
    Output : (B, T_out, leads)    e.g. (B, 490, 12)
    """
    ENC_CHANNELS  = 128
    DEC_CHANNELS  = 128
    ENC_DILATIONS = [1, 2, 4, 8, 16]   # 5 encoder blocks
    DEC_DILATIONS = [1, 2, 4, 8, 16]   # 5 decoder blocks

    def __init__(self, n_leads=12, input_len=490, horizon=490, dropout=0.1):
        super().__init__()
        C  = self.ENC_CHANNELS
        Cd = self.DEC_CHANNELS
        self.n_leads   = n_leads
        self.horizon   = horizon
        self.input_len = input_len

        # ── ENCODER ──────────────────────────────────────────────────────────
        # Project leads -> channels, then dilated residual blocks
        self.enc_input = nn.Sequential(
            nn.Conv1d(n_leads, C, 1, bias=False),
            nn.GroupNorm(8, C),
            nn.GELU()
        )
        self.enc_blocks = nn.ModuleList([
            ResBlock1D(C, d, kernel_size=5, dropout=dropout)
            for d in self.ENC_DILATIONS
        ])
        # Pool the entire time axis -> single context vector per sample
        self.enc_pool = nn.AdaptiveAvgPool1d(1)   # (B, C, T) -> (B, C, 1)

        # ── BOTTLENECK ───────────────────────────────────────────────────────
        # MLP to project context and expand to decoder length
        # We decode at a reduced temporal resolution first then upsample.
        # Reduced length = horizon // 8 (rounds down to 61 for horizon=490)
        self.dec_init_len = max(horizon // 8, 4)
        self.bottleneck = nn.Sequential(
            nn.Linear(C, Cd * self.dec_init_len),
            nn.GELU()
        )

        # ── DECODER ──────────────────────────────────────────────────────────
        self.dec_blocks = nn.ModuleList([
            ResBlock1D(Cd, d, kernel_size=5, dropout=dropout)
            for d in self.DEC_DILATIONS
        ])
        # Final projection: channels -> leads, at full resolution
        self.dec_out = nn.Conv1d(Cd, n_leads, 1)

    def forward(self, x):
        # x: (B, leads, T_in)
        B = x.size(0)

        # Encode
        h = self.enc_input(x)          # (B, C, T_in)
        for blk in self.enc_blocks:
            h = blk(h)                  # (B, C, T_in)
        ctx = self.enc_pool(h).squeeze(-1)  # (B, C)

        # Bottleneck: context -> decoder seed
        dec = self.bottleneck(ctx)      # (B, Cd * dec_init_len)
        dec = dec.view(B, self.DEC_CHANNELS, self.dec_init_len)  # (B, Cd, L_small)

        # Upsample to horizon
        dec = F.interpolate(dec, size=self.horizon, mode='linear', align_corners=False)
        # (B, Cd, horizon)

        for blk in self.dec_blocks:
            dec = blk(dec)

        out = self.dec_out(dec)         # (B, leads, horizon)
        return out.permute(0, 2, 1)     # (B, horizon, leads)


# Verify shapes
model_check = ECGEncoderDecoder(N_LEADS, INPUT_LEN, HORIZON).to(DEVICE)
with torch.no_grad():
    dummy = torch.randn(4, N_LEADS, INPUT_LEN).to(DEVICE)
    out   = model_check(dummy)
    assert out.shape == (4, HORIZON, N_LEADS), f"Shape error: {out.shape}"
    print(f"Forward pass : {tuple(dummy.shape)} -> {tuple(out.shape)}  OK")
del model_check
print("OK ECGEncoderDecoder defined")
print("   Encoder: dilated ResBlocks -> AdaptiveAvgPool -> global context")
print("   Decoder: MLP expand -> linear upsample -> dilated ResBlocks -> leads")
print("   Every output step sees the FULL encoded past (correct for forecasting)")


Forward pass : (4, 12, 490) -> (4, 490, 12)  OK
OK ECGEncoderDecoder defined
   Encoder: dilated ResBlocks -> AdaptiveAvgPool -> global context
   Decoder: MLP expand -> linear upsample -> dilated ResBlocks -> leads
   Every output step sees the FULL encoded past (correct for forecasting)


In [5]:
# CELL 5 - INSTANTIATE + SHAPE CHECK
model    = ECGEncoderDecoder(n_leads=N_LEADS, input_len=INPUT_LEN,
                              horizon=HORIZON, dropout=0.1).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

with torch.no_grad():
    dummy = torch.randn(4, N_LEADS, INPUT_LEN).to(DEVICE)
    out   = model(dummy)
    assert out.shape == (4, HORIZON, N_LEADS), f"Shape error: {out.shape}"
    print(f"Forward pass : {tuple(dummy.shape)} -> {tuple(out.shape)}")
    print(f"Output std (random init) : {out.std().item():.4f}  (want > 0.1)")

print(f"Parameters   : {n_params:,}")
print(f"Device       : {DEVICE}")
print("OK Model ready")


Forward pass : (4, 12, 490) -> (4, 490, 12)
Output std (random init) : 1.1744  (want > 0.1)
Parameters   : 2,654,092
Device       : cpu
OK Model ready


In [6]:
# CELL 6 - COMPOSITE LOSS: MAE + GRADIENT + FREQUENCY
#
# Three complementary terms:
#
# 1. MAE  (L1): robust per-sample amplitude matching. L1 over L2 because ECG
#    has large, sparse QRS spikes -- squared loss penalises those spikes
#    disproportionately and encourages the model to underestimate amplitude
#    to reduce the squared penalty (the v10 flatness problem).
#
# 2. Gradient L1: first-difference of prediction vs target. Penalises
#    flat output because a flat signal has ~zero gradient everywhere while
#    QRS complexes have large gradients. Weight raised from 2.0 (v11) -> 4.0
#    (v11 notebook warned to do this but shipped without the change).
#
# 3. FFT magnitude L1: L1 distance between |FFT(pred)| and |FFT(target)|
#    on the time axis. Forces the model to reproduce the RIGHT FREQUENCY
#    CONTENT -- specifically the ~1 Hz QRS band. A flat prediction has all
#    its energy at DC (0 Hz) while a real ECG has most energy at 1-40 Hz;
#    this term directly penalises that spectral mismatch.

GRAD_WEIGHT = 4.0
FFT_WEIGHT  = 1.0

class CompositeLoss(nn.Module):
    def __init__(self, grad_weight=GRAD_WEIGHT, fft_weight=FFT_WEIGHT):
        super().__init__()
        self.gw = grad_weight
        self.fw = fft_weight

    def forward(self, pred, target):
        # pred, target: (B, T, leads)

        # 1. MAE
        mae = F.l1_loss(pred, target)

        # 2. Gradient (first-difference)
        pred_d   = pred[:, 1:, :]   - pred[:, :-1, :]
        target_d = target[:, 1:, :] - target[:, :-1, :]
        grad_loss = F.l1_loss(pred_d, target_d)

        # 3. FFT magnitude (operate on time axis=1, averaged over leads)
        #    rfft returns complex; abs gives magnitude spectrum
        pred_fft   = torch.fft.rfft(pred,   dim=1).abs()
        target_fft = torch.fft.rfft(target, dim=1).abs()
        fft_loss   = F.l1_loss(pred_fft, target_fft)

        return mae + self.gw * grad_loss + self.fw * fft_loss


criterion = CompositeLoss()

# Sanity check: flat (constant-mean) prediction must be clearly worse than perfect
sample_t  = torch.from_numpy(y_train[:128].astype('float32'))
flat_pred = sample_t.mean(dim=1, keepdim=True).expand_as(sample_t)
flat_loss    = criterion(flat_pred, sample_t).item()
perfect_loss = criterion(sample_t, sample_t).item()
print(f"Loss if model predicted the per-sample MEAN (flat) : {flat_loss:.4f}")
print(f"Loss if model predicted PERFECTLY                  : {perfect_loss:.4f}")
print(f"Gap (flat is this much worse than perfect)          : {flat_loss - perfect_loss:.4f}")
print(f"CompositeLoss  grad_weight={GRAD_WEIGHT}  fft_weight={FFT_WEIGHT}")
if flat_loss - perfect_loss < 0.1:
    print("WARNING: flat prediction barely punished -- raise GRAD_WEIGHT further")
else:
    print("OK Flat predictions clearly penalised")


Loss if model predicted the per-sample MEAN (flat) : 14.2399
Loss if model predicted PERFECTLY                  : 0.0000
Gap (flat is this much worse than perfect)          : 14.2399
CompositeLoss  grad_weight=4.0  fft_weight=1.0
OK Flat predictions clearly penalised


In [7]:
# CELL 7 - TRAINING ENGINE
#
# Changes vs v11:
#   - max_lr  1.5e-4 -> 3e-4  (model wasn't learning fast enough)
#   - patience 10 -> 15       (encoder-decoder is slower to converge than shallow CNN;
#                              needs a bit more runway after the LR peak)
#   - Gradient clipping kept at 1.0
#   - Checkpoint name updated to v12

def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0.0
    for xb, yb in tqdm(loader, desc='Train', leave=False, ncols=88):
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item() * len(xb)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def eval_epoch(model, loader, device):
    model.eval()
    total_loss, preds, targets = 0.0, [], []
    for xb, yb in loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        pred = model(xb)
        total_loss += criterion(pred, yb).item() * len(xb)
        preds.append(pred.cpu().numpy())
        targets.append(yb.cpu().numpy())
    return (total_loss / len(loader.dataset),
            np.concatenate(preds), np.concatenate(targets))


def train_model(model, tr_loader, vl_loader,
                n_epochs=80, max_lr=3e-4, patience=15):
    steps_per_epoch = len(tr_loader)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=max_lr / 25,
        weight_decay=1e-4, eps=1e-8)

    scheduler = OneCycleLR(
        optimizer, max_lr=max_lr,
        epochs=n_epochs, steps_per_epoch=steps_per_epoch,
        pct_start=0.15,          # warmup shorter (15%) -- encoder converges faster
        anneal_strategy='cos',
        div_factor=25.0,
        final_div_factor=1e4)

    best_val, no_improve = float('inf'), 0
    ckpt    = os.path.join(CKPT_DIR, 'ECG_CNN_v12_best.pt')
    history = {'train_loss': [], 'val_loss': [], 'lr': []}

    sep = '-' * 72
    print(f'\n{sep}')
    print(f'  ECGEncoderDecoder v12  |  {INPUT_LEN/FS:.1f}s -> {HORIZON/FS:.1f}s  |  {n_params:,} params')
    print(f'  CompositeLoss(grad_w={GRAD_WEIGHT}, fft_w={FFT_WEIGHT})')
    print(f'  OneCycleLR(max={max_lr:.0e}, warmup=15%)  dropout=0.1  wd=1e-4')
    print(f'  batch={tr_loader.batch_size}  steps/ep={steps_per_epoch}  patience={patience}')
    print(sep)

    pbar = tqdm(range(1, n_epochs + 1), desc='Epochs', unit='ep', ncols=88)
    for ep in pbar:
        tr_loss       = train_epoch(model, tr_loader, optimizer, scheduler, DEVICE)
        vl_loss, _, _ = eval_epoch(model, vl_loader, DEVICE)
        cur_lr        = scheduler.get_last_lr()[0]

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['lr'].append(cur_lr)

        is_best = vl_loss < best_val
        if is_best:
            best_val = vl_loss; no_improve = 0
            torch.save(model.state_dict(), ckpt)
        else:
            no_improve += 1

        pbar.set_postfix(tr=f'{tr_loss:.4f}', vl=f'{vl_loss:.4f}',
                         lr=f'{cur_lr:.1e}', pat=no_improve)
        if ep % 5 == 0 or is_best or ep == 1:
            tqdm.write(
                f'  ep {ep:3d}  train={tr_loss:.5f}  val={vl_loss:.5f}'
                f'  lr={cur_lr:.2e}'
                f'{"  * best" if is_best else f"  (no-imp {no_improve}/{patience})"}')

        if no_improve >= patience:
            tqdm.write(f'  Early stop ep {ep}  best val={best_val:.6f}')
            break

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
    print(f'\n  Best checkpoint  val={best_val:.6f}  -> {ckpt}')
    print(f'{sep}\n')
    return history

print("OK Training engine ready")


OK Training engine ready


In [ ]:
# CELL 8 - TRAIN
#
# What to watch vs v11:
#   - Loss should drop substantially in the first 10 epochs (encoder is learning
#     to compress the past window). v11 barely moved from epoch 1.
#   - val_loss should track train_loss closely in early epochs, then diverge
#     slightly -- healthy overfitting signal, not the immediate flatline of v11.
#   - Variance check (Cell 14) should show pred_std tracking true_std. If still
#     flat on all leads, raise GRAD_WEIGHT to 6.0 in Cell 6.
#   - QRS F1 should be > 0.4 to confirm the model is tracking heartbeats.

history = train_model(model, cnn_tr, cnn_vl, n_epochs=80, max_lr=3e-4, patience=15)



------------------------------------------------------------------------
  ECGEncoderDecoder v12  |  4.9s -> 4.9s  |  2,654,092 params
  CompositeLoss(grad_w=4.0, fft_w=1.0)
  OneCycleLR(max=3e-04, warmup=15%)  dropout=0.1  wd=1e-4
  batch=64  steps/ep=490  patience=15
------------------------------------------------------------------------


Epochs:   0%|                                                    | 0/80 [00:00<?, ?ep/s]

Train:   0%|                                                    | 0/490 [00:00<?, ?it/s]

In [ ]:
# CELL 9 - TRAINING HISTORY
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(ep_range, history['train_loss'], color='#0ea5e9', lw=2,
             label='Train', marker='o', ms=3)
axes[0].plot(ep_range, history['val_loss'], color='#ef4444', lw=2,
             label='Val', marker='s', ms=3, ls='--')
best_ep = int(np.argmin(history['val_loss'])) + 1
axes[0].axvline(best_ep, color='gold', ls=':', lw=2, label=f'Best ep {best_ep}')
axes[0].set_title(f'ECGEncoderDecoder v12 — Loss  ({INPUT_LEN/FS:.1f}s -> {HORIZON/FS:.1f}s)',
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel(f'CompositeLoss (grad_w={GRAD_WEIGHT}, fft_w={FFT_WEIGHT})')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep_range, history['lr'], color='#10b981', lw=2, marker='o', ms=3)
axes[1].set_title('OneCycleLR (15% warmup)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Learning Rate')
axes[1].set_yscale('log'); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '01_training_history.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Best ep: {best_ep}  val loss: {min(history['val_loss']):.5f}")
print("OK Saved 01_training_history.png")


In [ ]:
# CELL 10 - TEST EVALUATION (de-normalised to mV, with PRD)
test_loss, test_preds_n, test_targets_n = eval_epoch(model, cnn_te, DEVICE)
print(f"Test CompositeLoss : {test_loss:.6f}  (normalised units)")

# De-normalise back to original mV scale
test_preds   = denormalize(test_preds_n)
test_targets = denormalize(test_targets_n)

mae_per_lead, rmse_per_lead, prd_per_lead = [], [], []
for i in range(N_LEADS):
    p, t = test_preds[:,:,i].flatten(), test_targets[:,:,i].flatten()
    mae_per_lead.append(mean_absolute_error(t, p))
    rmse_per_lead.append(np.sqrt(mean_squared_error(t, p)))
    # PRD = percent root-mean-square difference (clinical standard)
    # <10% excellent, 10-25% good, >25% poor
    prd = 100.0 * np.sqrt(np.sum((t - p) ** 2) / (np.sum(t ** 2) + 1e-9))
    prd_per_lead.append(prd)

mae_macro  = np.mean(mae_per_lead)
rmse_macro = np.mean(rmse_per_lead)
prd_macro  = np.mean(prd_per_lead)

print(f"\n{'Lead':>6s}   {'MAE(mV)':>8s}   {'RMSE(mV)':>9s}   {'PRD(%)':>7s}")
print('-' * 38)
for i, name in enumerate(LEAD_NAMES):
    flag = '  *** FAIL' if prd_per_lead[i] > 50 else ('  OK' if prd_per_lead[i] < 25 else '')
    print(f"  {name:>4s}   {mae_per_lead[i]:8.4f}   {rmse_per_lead[i]:9.4f}   {prd_per_lead[i]:7.2f}{flag}")
print('-' * 38)
print(f"  {'Macro':>4s}   {mae_macro:8.4f}   {rmse_macro:9.4f}   {prd_macro:7.2f}")
print(f"\nPRD interpretation: <10%% excellent | 10-25%% good | >25%% poor")
if prd_macro > 50:
    print("FAIL: PRD > 50% -- model still not tracking signal morphology.")
    print("  Next step: raise GRAD_WEIGHT to 6.0 in Cell 6 and retrain.")
elif prd_macro > 25:
    print("Partial: PRD 25-50% -- some morphology captured but needs improvement.")
else:
    print("OK: PRD < 25% -- model is capturing ECG morphology.")
print("OK Evaluation complete (metrics in original mV scale)")


In [ ]:
# CELL 11 - PER-LEAD RMSE + PRD BAR CHARTS
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, N_LEADS))
bars = axes[0].bar(np.arange(N_LEADS), rmse_per_lead, width=0.62,
                   color=colors, alpha=0.88, edgecolor='black', lw=0.5)
axes[0].axhline(rmse_macro, color='#facc15', ls='--', lw=2.5,
                label=f'Macro RMSE: {rmse_macro:.4f} mV')
for bar, val in zip(bars, rmse_per_lead):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_xticks(np.arange(N_LEADS)); axes[0].set_xticklabels(LEAD_NAMES, fontsize=10, fontweight='bold')
axes[0].set_ylabel('RMSE (mV)'); axes[0].set_title('Per-Lead RMSE', fontsize=13, fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3, axis='y')

bars2 = axes[1].bar(np.arange(N_LEADS), prd_per_lead, width=0.62,
                    color=colors, alpha=0.88, edgecolor='black', lw=0.5)
axes[1].axhline(prd_macro, color='#facc15', ls='--', lw=2.5,
                label=f'Macro PRD: {prd_macro:.1f}%')
axes[1].axhline(10,  color='green',  ls=':', lw=1.5, label='Excellent (10%)')
axes[1].axhline(25,  color='orange', ls=':', lw=1.5, label='Good threshold (25%)')
for bar, val in zip(bars2, prd_per_lead):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[1].set_xticks(np.arange(N_LEADS)); axes[1].set_xticklabels(LEAD_NAMES, fontsize=10, fontweight='bold')
axes[1].set_ylabel('PRD (%)'); axes[1].set_title('Per-Lead PRD (clinical metric)', fontsize=13, fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '02_per_lead_rmse_prd.png'), dpi=150, bbox_inches='tight')
plt.show()
print("OK Saved 02_per_lead_rmse_prd.png")


In [ ]:
# CELL 12 - PREDICTED vs ACTUAL (de-normalised mV)
N_ROWS       = 4
lead_indices = [0, 1, 2, 6]   # I, II, III, V1
t_axis       = np.arange(HORIZON) / FS

fig, axes = plt.subplots(N_ROWS, 4, figsize=(20, 14))
for row in range(N_ROWS):
    for col, li in enumerate(lead_indices):
        ax     = axes[row, col]
        actual = test_targets[row, :, li]
        pred   = test_preds[row,   :, li]
        rmse_i = np.sqrt(mean_squared_error(actual, pred))
        ax.plot(t_axis, actual, color='#0ea5e9', lw=1.8, label='Actual',    alpha=0.92)
        ax.plot(t_axis, pred,   color='#ef4444', lw=1.4, label='Predicted', alpha=0.88, ls='--')
        ax.set_title(f'{LEAD_NAMES[li]}  RMSE={rmse_i:.3f} mV',
                     fontsize=10, fontweight='bold')
        ax.set_xlabel('Time (s)', fontsize=8); ax.set_ylabel('mV', fontsize=8)
        ax.grid(True, alpha=0.25)
        if col == 0 and row == 0:
            ax.legend(fontsize=8, loc='upper right')

fig.suptitle(f'ECGEncoderDecoder v12: Predicted vs Actual — {HORIZON/FS:.1f}s horizon'
             f'\nBlue = Actual  |  Red = Predicted',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '03_predictions_overlay.png'), dpi=150, bbox_inches='tight')
plt.show()
print("OK Saved 03_predictions_overlay.png")


In [ ]:
# CELL 13 - QRS-DETECTION F1
#
# A flat prediction produces zero detected peaks -> F1=0, impossible to fake.
# v11 got F1=0.10 (FAIL). Target: F1 > 0.4 for this to be considered learning.

TOL_SAMPLES = int(0.05 * FS)   # 50ms tolerance window

def detect_peaks(sig, min_height_frac=0.4):
    height = min_height_frac * sig.std() + sig.mean()
    peaks, _ = find_peaks(sig, height=height, distance=int(0.25 * FS))
    return peaks

def qrs_f1(actual, pred, tol=TOL_SAMPLES):
    tp = fp = fn = 0
    for n in range(actual.shape[0]):
        a_peaks = detect_peaks(actual[n])
        p_peaks = detect_peaks(pred[n])
        matched_a = set()
        for pp in p_peaks:
            dists = np.abs(a_peaks - pp) if len(a_peaks) else np.array([])
            if len(dists) and dists.min() <= tol:
                idx = a_peaks[np.argmin(dists)]
                if idx not in matched_a:
                    matched_a.add(idx); tp += 1
                else:
                    fp += 1
            else:
                fp += 1
        fn += len(a_peaks) - len(matched_a)
    precision = tp / (tp + fp + 1e-9)
    recall    = tp / (tp + fn + 1e-9)
    f1        = 2 * precision * recall / (precision + recall + 1e-9)
    return precision, recall, f1, tp, fp, fn

li = LEAD_NAMES.index('II') if 'II' in LEAD_NAMES else 1
n_eval = min(300, test_targets.shape[0])
precision, recall, f1, tp, fp, fn = qrs_f1(
    test_targets[:n_eval, :, li], test_preds[:n_eval, :, li])

print(f"QRS-detection on Lead {LEAD_NAMES[li]}  (n={n_eval} windows, tol=±50ms)")
print(f"  True positives  : {tp}")
print(f"  False positives : {fp}")
print(f"  False negatives : {fn}")
print(f"  Precision       : {precision:.3f}")
print(f"  Recall          : {recall:.3f}")
print(f"  F1              : {f1:.3f}")
if f1 < 0.3:
    print("FAIL -- model is not resolving individual heartbeats")
    print("  -> If PRD also > 50%, architecture is wrong. If PRD is improving,")
    print("     raise GRAD_WEIGHT to 6.0 and retrain.")
elif f1 < 0.6:
    print("Partial -- model detects some beats but misses or hallucinates many")
else:
    print("Good -- model is tracking real R-peaks")


In [ ]:
# CELL 14 - PREDICTION VARIANCE CHECK
#
# v11 showed under-variance on ALL 12 leads (ratio < 0.5 everywhere).
# Target: pred_std / true_std > 0.6 on most leads after v12 fixes.

pred_std = test_preds.std(axis=0)
true_std = test_targets.std(axis=0)
t_axis   = np.arange(HORIZON) / FS

fig, axes = plt.subplots(3, 4, figsize=(18, 10))
flat_leads = []
for i, (ax, name) in enumerate(zip(axes.flatten(), LEAD_NAMES)):
    ax.plot(t_axis, true_std[:,i], color='#0ea5e9', lw=1.4, label='Actual std')
    ax.plot(t_axis, pred_std[:,i], color='#ef4444', lw=1.4, label='Pred std', ls='--')
    ax.set_title(f'Lead {name}', fontweight='bold', fontsize=10)
    ax.set_xlabel('Time (s)', fontsize=8); ax.set_ylabel('Std (mV)', fontsize=8)
    ax.grid(alpha=0.25)
    if i == 0: ax.legend(fontsize=8)
    ratio = pred_std[:,i].mean() / (true_std[:,i].mean() + 1e-9)
    if ratio < 0.5:
        ax.set_facecolor('#fff0f0'); flat_leads.append(name)
        ax.set_title(f'Lead {name} — LOW VAR ({ratio:.2f}x)', color='red', fontweight='bold')
    else:
        ax.set_title(f'Lead {name} ({ratio:.2f}x)', fontweight='bold', fontsize=10)

fig.suptitle('ECGEncoderDecoder v12 — Prediction Variance Check', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '04_variance_check.png'), dpi=150, bbox_inches='tight')
plt.show()

if flat_leads:
    print(f"WARNING: Still under-variance leads: {flat_leads}")
    if len(flat_leads) == N_LEADS:
        print("  ALL leads under-variance -> architecture problem persists.")
        print("  Check that X and y in your data are truly past/future windows")
        print("  (not the same window, which would make this a reconstruction task).")
    else:
        print("  Partial fix: raise GRAD_WEIGHT to 6.0 in Cell 6.")
else:
    overall_ratio = pred_std.mean() / true_std.mean()
    print(f"OK Prediction variance tracks actual variance. Overall ratio = {overall_ratio:.2f}x")


In [ ]:
# CELL 15 - SAVE RESULTS
results = {
    'model'         : 'ECGEncoderDecoder_v12',
    'horizon_s'     : HORIZON / FS,
    'input_s'       : INPUT_LEN / FS,
    'n_parameters'  : n_params,
    'test_loss'     : float(test_loss),
    'mae_per_lead'  : mae_per_lead,
    'mae_macro'     : float(mae_macro),
    'rmse_per_lead' : rmse_per_lead,
    'rmse_macro'    : float(rmse_macro),
    'prd_per_lead'  : prd_per_lead,
    'prd_macro'     : float(prd_macro),
    'qrs_precision' : float(precision),
    'qrs_recall'    : float(recall),
    'qrs_f1'        : float(f1),
    'grad_weight'   : GRAD_WEIGHT,
    'fft_weight'    : FFT_WEIGHT,
    'norm_mu'       : NORM_MU,
    'norm_sigma'    : NORM_SIGMA,
    'history'       : history,
    'lead_names'    : LEAD_NAMES,
    'test_preds'    : test_preds,
    'test_targets'  : test_targets,
}
path = os.path.join(CKPT_DIR, 'ECG_CNN_v12_results.pkl')
with open(path, 'wb') as f:
    pickle.dump(results, f)

print(f'\n{"="*62}')
print(f'  ECGEncoderDecoder v12 FINAL  ({INPUT_LEN/FS:.1f}s -> {HORIZON/FS:.1f}s)')
print(f'{"="*62}')
print(f'  Parameters     : {n_params:,}')
print(f'  Test loss      : {test_loss:.6f}  (CompositeLoss, normalised)')
print(f'  Macro MAE      : {mae_macro:.6f} mV')
print(f'  Macro RMSE     : {rmse_macro:.6f} mV')
print(f'  Macro PRD      : {prd_macro:.2f}%   (<10%% excellent, <25%% good)')
print(f'  QRS F1         : {f1:.3f}  (precision={precision:.3f}, recall={recall:.3f})')
print(f'{"="*62}')
print(f'  Checkpoint : {CKPT_DIR}/ECG_CNN_v12_best.pt')
print(f'  Results    : {path}')
print('OK ECGEncoderDecoder v12 complete')
